In [ ]:
!pip install datasets transformers trl flash-attn --no-build-isolation

In [ ]:
import math
from typing import Dict, List
from copy import deepcopy
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence
from dataclasses import dataclass
from datasets import load_dataset
from trl import DPOTrainer, DPOConfig
from transformers import AutoTokenizer, AutoModelForCausalLM

# 1) Introduction to Direct Preference Optimization (DPO)

![](https://miro.medium.com/v2/resize:fit:1400/1*AqKOT0pxzi5kOgiobb-Fvg.png)


---

After SFT, a model can follow instructions — but it might still produce responses that are **unhelpful, verbose, or misaligned** with human preferences. DPO is the alignment step that fixes this.

> **The LLM training pipeline:**
>
> **Pretraining** (learn language) → **SFT** (learn to follow instructions) → **DPO** (learn which responses humans prefer)

---

### The Core Idea

DPO trains on **(chosen, rejected) pairs** — for the same prompt, a human labels one response as better than the other:
```
Prompt:    "Explain gravity in one sentence."
Chosen:    "Gravity is the force that pulls objects with mass toward each other."  ✅
Rejected:  "It's just how things fall down because of science stuff."              ❌
```

The model learns to **increase the probability of chosen responses** and **decrease the probability of rejected ones** — but only relative to a frozen **reference model** to prevent the model from drifting too far.

---

### Why DPO instead of RLHF?

| | **RLHF** | **DPO** |
|--|----------|---------|
| **Steps** | Train reward model → then use PPO to optimize policy | Single training step directly on preference pairs |
| **Complexity** | Needs reward model + RL loop + value head | Just a modified loss function on the same LM |
| **Stability** | PPO can be unstable, many hyperparams | More stable, fewer hyperparams |
| **Result** | Same goal | Same goal, simpler path |

> 💡 DPO skips the reward model entirely. It mathematically shows that you can **implicitly** optimize the same objective as RLHF by directly training on preference pairs with a clever loss function.

---

### The DPO Loss (Intuition)
```
loss = -log(sigmoid(β × [(policy_chosen - policy_rejected) - (ref_chosen - ref_rejected)]))
                          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
                          "Does our model prefer chosen?"    "Did the ref model already prefer it?"
```

> The loss asks: **"Is our model learning a stronger preference for the chosen response than the reference model had?"** If yes, loss is low. If not, loss pushes the model to fix that.
>
> **β (beta)** controls regularization — how far the policy can drift from the reference. Typical values: `0.1 - 0.5`.

---

### What this notebook covers:

| Topic | Description |
|-------|-------------|
| **Quick-start with TRL** | Using `DPOTrainer` for DPO in a few lines |
| **Tokenization** | Processing (prompt, chosen, rejected) triplets with prompt masking |
| **Reference Model** | Frozen copy of the policy used as a KL anchor |
| **Log-probability Extraction** | Computing per-sequence log-probs from logits |
| **DPO Loss Function** | Sigmoid, hinge, and IPO loss variants |
| **From-scratch Training Loop** | Full DPO pipeline without TRL |

In [ ]:
model_name = "Qwen/Qwen2.5-0.5B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2-0.5B-Instruct")

# Load a standard preference dataset with chosen/rejected pairs
train_dataset = load_dataset("trl-lib/ultrafeedback_binarized", split="train")
print(train_dataset, "\n")

for key in train_dataset[0].keys():
    print(f"{key}: ", train_dataset[0][key])

Dataset({
    features: ['chosen', 'rejected', 'score_chosen', 'score_rejected'],
    num_rows: 62135
}) 

chosen:  [{'content': 'Use the pygame library to write a version of the classic game Snake, with a unique twist', 'role': 'user'}, {'content': "Sure, I'd be happy to help you write a version of the classic game Snake using the pygame library! Here's a basic outline of how we can approach this:\n\n1. First, we'll need to set up the game display and create a game object that we can use to handle the game's state.\n2. Next, we'll create the game's grid, which will be used to represent the game board. We'll need to define the size of the grid and the spaces within it.\n3. After that, we'll create the snake object, which will be used to represent the player's movement. We'll need to define the size of the snake and the speed at which it moves.\n4. We'll also need to create a food object, which will be used to represent the food that the player must collect to score points. We'll need t

In [ ]:
training_args = DPOConfig(output_dir="Qwen2-0.5B-DPO", logging_steps=10)
trainer = DPOTrainer(
    model=model,
    args=training_args,
    processing_class=tokenizer,
    train_dataset=train_dataset,
)
trainer.train()

# 2) DPO from Scratch — Deep Dive

In [ ]:
model_name = "Qwen/Qwen2.5-0.5B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2-0.5B-Instruct")

## 2.1 Data Processing

In [ ]:
class TinyDPODataset(torch.utils.data.Dataset):
    """A minimal paired-preference dataset with dict fields: prompt, chosen, rejected.

    This avoids external downloads; perfect for quick CPU demos.
    """

    def __init__(self):
        self.samples = [
            {
                "prompt": "Explain what DPO is, briefly.",
                "chosen": " Direct Preference Optimization aligns a policy to preferences using a reference policy.",
                "rejected": " It's just supervised fine-tuning on random text.",
            },
            {
                "prompt": "What does beta do in DPO?",
                "chosen": " It controls how much the policy deviates from the reference model (regularization).",
                "rejected": " It increases the sequence length for generation.",
            },
            {
                "prompt": "Why use a reference model in DPO?",
                "chosen": " To anchor the policy and prevent reward hacking by comparing likelihood ratios.",
                "rejected": " To provide labels for supervised learning of token classes.",
            },
            {
                "prompt": "How are losses computed in DPO?",
                "chosen": " On log-prob differences of chosen vs rejected, adjusted by reference differences.",
                "rejected": " By maximizing cross-entropy of the prompt tokens only.",
            },
        ]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

# === Example ===
dataset = TinyDPODataset()
sample = next(iter(dataset.samples))
for k, v in sample.items():
    print(f"{k}: {v}")

prompt: Explain what DPO is, briefly.
chosen:  Direct Preference Optimization aligns a policy to preferences using a reference policy.
rejected:  It's just supervised fine-tuning on random text.


## 2.2 Tokenization


> For DPO we need to tokenize prompt+chosen and prompt+rejected separately, then mask out prompt tokens in labels so the loss is computed only on the completion region.

In [ ]:
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id
if tokenizer.model_max_length > 100_000:
    tokenizer.model_max_length = 2048

def build_tokenized_answer(tokenizer, prompt: str, answer: str):
    """Tokenize prompt+answer and split into prompt vs. answer token regions.

    Strategy:
    - Tokenize prompt alone and prompt+answer together
    - Slice answer tokens by removing first len(prompt_tokens) from full tokens
    - Handle edge case where last prompt token merges with first answer token
    """
    full = tokenizer(prompt + answer, add_special_tokens=False)
    prompt_only = tokenizer(prompt, add_special_tokens=False)

    # Find where the answer tokens start
    # Handle possible last-token merge: if prefix doesn't match, shift back by 1
    start = len(prompt_only["input_ids"])
    if prompt_only["input_ids"] != full["input_ids"][:start]:
        start -= 1

    return {
        "prompt_input_ids": full["input_ids"][:start],              # (prompt_len,)
        "prompt_attention_mask": full["attention_mask"][:start],     # (prompt_len,)
        "input_ids": full["input_ids"][start:],                     # (answer_len,)
        "attention_mask": full["attention_mask"][start:],            # (answer_len,)
    }

In [ ]:
def tokenize_row(
    feature: Dict[str, str],
    tokenizer,
    max_length: int = 512,
    max_prompt_length: int = 256,
    max_completion_length: int = 256,
    truncation_mode: str = "keep_end",
) -> Dict[str, List[int]]:
    """Tokenize a single DPO row (prompt, chosen, rejected) into padded sequences with labels.

    Key steps:
    1. Tokenize prompt+chosen and prompt+rejected
    2. Add BOS to prompt, EOS to completions
    3. Truncate prompt then completions if exceeding max_length
    4. Mask prompt tokens in labels with -100 (ignored in loss)
    """
    prompt_tok = tokenizer(feature["prompt"], add_special_tokens=False)
    chosen_tok = build_tokenized_answer(tokenizer, feature["prompt"], feature["chosen"])
    rejected_tok = build_tokenized_answer(tokenizer, feature["prompt"], feature["rejected"])

    # Add BOS token to the start of prompt region
    # Input: (prompt_len,) -> Output: (prompt_len + 1,)
    bos_id = getattr(tokenizer, "bos_token_id", None)
    if bos_id is not None:
        prompt_tok = {
            "prompt_input_ids": [bos_id] + prompt_tok["input_ids"],
            "prompt_attention_mask": [1] + prompt_tok["attention_mask"],
        }
        for toks in (chosen_tok, rejected_tok):
            toks["prompt_input_ids"] = [bos_id] + toks["prompt_input_ids"]
            toks["prompt_attention_mask"] = [1] + toks["prompt_attention_mask"]
    else:
        prompt_tok = {
            "prompt_input_ids": prompt_tok["input_ids"],
            "prompt_attention_mask": prompt_tok["attention_mask"],
        }

    # Ensure EOS at end of completions
    # Input: (answer_len,) -> Output: (answer_len + 1,) if EOS was missing
    eos_id = tokenizer.eos_token_id
    for toks in (chosen_tok, rejected_tok):
        if not toks["input_ids"] or toks["input_ids"][-1] != eos_id:
            toks["input_ids"].append(eos_id)
            toks["attention_mask"].append(1)

    # --- Truncation ---
    # If prompt + longest_completion exceeds max_length, truncate prompt first
    longest_comp = max(len(chosen_tok["input_ids"]), len(rejected_tok["input_ids"]))
    limit = max_prompt_length or (max_length // 2)

    for toks in (prompt_tok, chosen_tok, rejected_tok):
        if len(toks["prompt_input_ids"]) + longest_comp > max_length:
            if truncation_mode == "keep_start":
                toks["prompt_input_ids"] = toks["prompt_input_ids"][:limit]
                toks["prompt_attention_mask"] = toks["prompt_attention_mask"][:limit]
            else:  # keep_end — keep the most recent context
                toks["prompt_input_ids"] = toks["prompt_input_ids"][-limit:]
                toks["prompt_attention_mask"] = toks["prompt_attention_mask"][-limit:]

    # If still too long, truncate completions
    for toks in (chosen_tok, rejected_tok):
        total = len(toks["prompt_input_ids"]) + len(toks["input_ids"])
        if total > max_length:
            cap = min(max_completion_length, max_length - len(toks["prompt_input_ids"]))
            toks["input_ids"] = toks["input_ids"][:cap]
            toks["attention_mask"] = toks["attention_mask"][:cap]

    # --- Build final sequences with masked labels ---
    # Prompt tokens get label=-100 so they are ignored in loss
    # Input: prompt (prompt_len,) + completion (comp_len,) -> Output: (seq_len,)
    def build_sequence(toks):
        ids = toks["prompt_input_ids"] + toks["input_ids"]
        mask = toks["prompt_attention_mask"] + toks["attention_mask"]
        labels = ids[:]
        labels[: len(toks["prompt_input_ids"])] = [-100] * len(toks["prompt_input_ids"])
        return ids, mask, labels

    c_ids, c_mask, c_labels = build_sequence(chosen_tok)
    r_ids, r_mask, r_labels = build_sequence(rejected_tok)

    return {
        "prompt_input_ids": prompt_tok["prompt_input_ids"],
        "prompt_attention_mask": prompt_tok["prompt_attention_mask"],
        "chosen_input_ids": c_ids,
        "chosen_attention_mask": c_mask,
        "chosen_labels": c_labels,
        "rejected_input_ids": r_ids,
        "rejected_attention_mask": r_mask,
        "rejected_labels": r_labels,
    }


In [ ]:
tokenized_rows = [
    tokenize_row(
        ex,
        tokenizer=tokenizer,
        max_length=256,
        max_prompt_length=128,
        max_completion_length=128,
        truncation_mode="keep_end",
    )
    for ex in dataset
]

for k, v in tokenized_rows[1].items():
    print(k, '\n', v)
    print('\n')

prompt_input_ids 
 [3838, 1558, 13440, 653, 304, 422, 2045, 30]


prompt_attention_mask 
 [1, 1, 1, 1, 1, 1, 1, 1]


chosen_input_ids 
 [3838, 1558, 13440, 653, 304, 422, 2045, 30, 1084, 11574, 1246, 1753, 279, 4842, 3483, 42298, 504, 279, 5785, 1614, 320, 22308, 2022, 568, 151643]


chosen_attention_mask 
 [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


chosen_labels 
 [-100, -100, -100, -100, -100, -100, -100, -100, 1084, 11574, 1246, 1753, 279, 4842, 3483, 42298, 504, 279, 5785, 1614, 320, 22308, 2022, 568, 151643]


rejected_input_ids 
 [3838, 1558, 13440, 653, 304, 422, 2045, 30, 1084, 12703, 279, 8500, 3084, 369, 9471, 13, 151643]


rejected_attention_mask 
 [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


rejected_labels 
 [-100, -100, -100, -100, -100, -100, -100, -100, 1084, 12703, 279, 8500, 3084, 369, 9471, 13, 151643]




## 2.3 DPO Data Collator

> Pads variable-length sequences in a batch to uniform length.
> Prompt fields are left-padded (so the prompt aligns at the right edge); completion fields are right-padded.

In [ ]:
PAD_VALUES = {"_input_ids": "pad", "_labels": -100, "_attention_mask": 0}


@dataclass
class DPODataCollatorWithPadding:
    pad_token_id: int = 0
    label_pad_token_id: int = -100

    def __call__(self, features: List[Dict]) -> Dict[str, torch.Tensor]:
        """Pad all sequences in the batch to the maximum length per key.

        Input: list of dicts, each with (varying_len,) tensors
        Output: dict of tensors, each (batch_num, max_seq_len)
        """
        padded: Dict[str, torch.Tensor] = {}
        for k in features[0]:
            if not k.endswith(("_input_ids", "_attention_mask", "_labels")):
                continue

            seqs = [torch.tensor(ex[k], dtype=torch.long) for ex in features]

            # Choose pad value based on field type
            pad_val = (
                self.pad_token_id if k.endswith("_input_ids")
                else self.label_pad_token_id if k.endswith("_labels")
                else 0
            )

            # Left-pad prompt fields by reversing, padding, then reversing back
            is_prompt = k.startswith("prompt_")
            if is_prompt:
                seqs = [s.flip(0) for s in seqs]

            # Input: list of (varying_len,) -> Output: (batch_num, max_seq_len)
            tensor = pad_sequence(seqs, batch_first=True, padding_value=pad_val)

            if is_prompt:
                tensor = tensor.flip(1)

            padded[k] = tensor

        return padded

In [ ]:
# Test the collator
collator = DPODataCollatorWithPadding()
test_set = collator(tokenized_rows)
for k, v in test_set.items():
    print(k, "\n", v, "\n")

prompt_input_ids 
 tensor([[  840, 20772,  1128,   422,  2045,   374,    11, 26753,    13],
        [    0,  3838,  1558, 13440,   653,   304,   422,  2045,    30],
        [10234,   990,   264,  5785,  1614,   304,   422,  2045,    30],
        [    0,  4340,   525, 17683, 24182,   304,   422,  2045,    30]]) 

prompt_attention_mask 
 tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1],
        [0, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1],
        [0, 1, 1, 1, 1, 1, 1, 1, 1]]) 

chosen_input_ids 
 tensor([[   840,  20772,   1128,    422,   2045,    374,     11,  26753,     13,
           7139,  48805,  57739,   5285,     82,    264,   4842,    311,  19322,
           1667,    264,   5785,   4842,     13, 151643,      0],
        [  3838,   1558,  13440,    653,    304,    422,   2045,     30,   1084,
          11574,   1246,   1753,    279,   4842,   3483,  42298,    504,    279,
           5785,   1614,    320,  22308,   2022,    568, 151643],
        [ 10234,    990,    264,   5

## 2.4 Model Loading

DPO requires two models:
- Policy model (trainable): the model we are aligning
- Reference model (frozen): a copy used to compute the KL anchor

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_name, torch_dtype="auto", device_map="cuda", trust_remote_code=True
)

# Create frozen reference model (deepcopy with all gradients disabled)
ref_model = deepcopy(model)
for p in ref_model.parameters():
    p.requires_grad = False

# Disable dropout in both models for deterministic log-prob computation
def disable_dropout(m: torch.nn.Module):
    for module in m.modules():
        if isinstance(module, torch.nn.Dropout):
            module.p = 0

disable_dropout(model)
disable_dropout(ref_model)

config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

## 2.5 Forward Utils

> Core functions for computing per-sequence log-probabilities and running efficient concatenated forward passes.

# 🔧 DPO Forward Pass Utilities — How Log-Probabilities Are Computed

---

DPO needs to answer one question: **"How likely does the model think the chosen vs rejected response is?"** This is measured using **log-probabilities**. The utilities below handle this computation efficiently.

---

## Step 1: `get_batch_logps` — Extract log-probabilities from logits

The model outputs **logits** (raw scores for every token in the vocabulary). We need to extract the log-probability of the **actual tokens** that were in the chosen/rejected response.
```python
# Model outputs logits for every possible next token:
logits shape: (batch, seq_len, vocab_size)    # e.g., (2, 10, 151936)

# We only care about the probability assigned to the ACTUAL next token:
labels shape: (batch, seq_len)                # e.g., (2, 10)
```

### The process:
```
Step 1: Shift (same right-shift as in training loss)
  logits: [pred_B, pred_C, pred_D, pred_E]     # logits[:-1]
  labels: [B,      C,      D,      E]           # labels[1:]

Step 2: log_softmax → convert logits to log-probabilities
  log_probs: (batch, seq_len-1, vocab_size)

Step 3: gather → pick the log-prob of each actual token
  Example: if label token is "C" (id=42), pick log_probs[:, :, 42]
  per_token_logps: (batch, seq_len-1)

Step 4: mask prompt tokens (label == -100) and sum
  [-100, -100, -100, 0.3, 0.5, 0.2]    ← per-token log-probs
   ^^^^^^^^^^^^^^^^^^^^                     prompt (masked, ignored)
                      ^^^^^^^^^^^^^^^^^     response (summed)
  Result: 0.3 + 0.5 + 0.2 = 1.0        ← one number per sequence
  Output: (batch,)
```

> 🔑 The output is a **single number per sequence** — the total log-probability of the response tokens. Higher = model thinks this response is more likely.

---

## Step 2: `concatenated_inputs` — Stack chosen + rejected into one batch

Instead of running the model **twice** (once for chosen, once for rejected), we **concatenate** them into a single batch and run the model **once**. This is purely for efficiency.
```
Before concatenation:
  chosen_input_ids:    (batch, seq_len_chosen)     e.g., (4, 20)
  rejected_input_ids:  (batch, seq_len_rejected)   e.g., (4, 25)

After concatenation:
  concatenated_input_ids:  (2*batch, max_seq_len)  e.g., (8, 25)
                            ^^^^^^^^
                            first 4 = chosen, last 4 = rejected

  Shorter sequences padded to max_seq_len:
  chosen  (len 20): [tok, tok, ..., tok, PAD, PAD, PAD, PAD, PAD]  → padded to 25
  rejected (len 25): [tok, tok, ..., tok, tok, tok, tok, tok, tok]  → already 25
```

---

## Step 3: `concatenated_forward` — Single forward pass for both chosen and rejected

Puts it all together:
```
Input batch:
  chosen_input_ids:   (batch, seq_c)
  rejected_input_ids: (batch, seq_r)
        │
        ▼
  concatenated_inputs()
  → concatenated_input_ids: (2*batch, max_seq)
        │
        ▼
  model.forward()
  → logits: (2*batch, max_seq, vocab_size)
        │
        ▼
  get_batch_logps()
  → logps: (2*batch,)
        │
        ▼
  Split at midpoint
  → chosen_logps:  (batch,)    ← first half
  → rejected_logps: (batch,)   ← second half
```

> 💡 **Why this matters for DPO:** The DPO loss needs four numbers per example — `policy_chosen_logps`, `policy_rejected_logps`, `ref_chosen_logps`, `ref_rejected_logps`. This function is called **twice** — once for the policy model (with gradients), once for the reference model (frozen, no gradients) — to get all four.

---

## 📊 Dimension Summary

| Function | Input | Output |
|----------|-------|--------|
| `get_batch_logps` | logits `(batch, seq, vocab)` + labels `(batch, seq)` | `(batch,)` — summed log-prob per sequence |
| `concatenated_inputs` | chosen `(batch, seq_c)` + rejected `(batch, seq_r)` | concatenated `(2*batch, max_seq)` |
| `concatenated_forward` | batch with chosen + rejected | `chosen_logps (batch,)`, `rejected_logps (batch,)`, logits for both |

In [ ]:
def pad_to_length(t: torch.Tensor, length: int, pad_value: int, dim: int = -1) -> torch.Tensor:
    """Pad tensor along `dim` to reach `length`.

    Input: (*, current_len, *) -> Output: (*, length, *)
    """
    if t.size(dim) >= length:
        return t
    pad_shape = list(t.shape)
    pad_shape[dim] = length - t.size(dim)
    return torch.cat([t, torch.full(pad_shape, pad_value, dtype=t.dtype, device=t.device)], dim=dim)


def get_batch_logps(
    logits: torch.Tensor,
    labels: torch.Tensor,
    label_pad_token_id: int = -100,
    # <-- NEW: ORPO needs this set to True
    average_log_prob: bool = False,
) -> torch.Tensor:
    """Compute summed log-probabilities of target tokens per sequence.

    For causal LMs, logits predict the NEXT token, so we shift:
    - logits[:, :-1, :] predicts token at position 1, 2, ..., T-1
    - labels[:, 1:]     provides the ground truth for those positions

    Input logits:  (batch_num, seq_len, vocab_size)
    Input labels:  (batch_num, seq_len)
    Output:        (batch_num,) — summed log-prob per sequence
    """
    # Shift: align logits with next-token labels
    # ((batch_num, seq_len-1, vocab_size)
    labels = labels[:, 1:].clone()
    logits = logits[:, :-1, :]

    # Mask out padding/prompt tokens (label == -100)
    # (batch_num, seq_len-1)
    loss_mask = labels != label_pad_token_id
    safe_labels = labels.clone()
    safe_labels[~loss_mask] = 0  # safe index for gather; masked out later

    # Gather log-probs at target token indices
    # safe_labels is the actual label from rejected/accepted answer

    # Input: log_softmax (batch_num, seq_len-1, vocab_size), index (batch_num, seq_len-1, 1)
    # Output: (batch_num, seq_len-1)
    per_token_logps = torch.gather(
        F.log_softmax(logits, dim=-1), dim=2, index=safe_labels.unsqueeze(2)
    ).squeeze(2)

    # Mask and aggregate
    # (batch_num, seq_len-1) -> (batch_num,)
    masked_logps = per_token_logps * loss_mask

    if average_log_prob:
        # ORPO: divide by number of valid tokens per sequence
        # This keeps values in a reasonable range for the odds computation
        return masked_logps.sum(-1) / loss_mask.sum(-1).clamp_min(1)
    else:
        # DPO: sum over all valid tokens
        return masked_logps.sum(-1)


def concatenated_inputs(
    batch: Dict[str, torch.Tensor],
    pad_token_id: int,
    label_pad_token_id: int = -100,
    device: torch.device | None = None,
) -> Dict[str, torch.Tensor]:
    """Stack chosen and rejected sequences into one batch for a single forward pass.

    Chosen sequences go first, rejected second.
    Input:  chosen (batch_num, seq_len_c), rejected (batch_num, seq_len_r)
    Output: concatenated (2*batch_num, max(seq_len_c, seq_len_r))
    """
    device = device or next(iter(batch.values())).device
    max_len = max(batch["chosen_input_ids"].size(1), batch["rejected_input_ids"].size(1))

    suffix_pad = {"input_ids": pad_token_id, "attention_mask": 0, "labels": label_pad_token_id}
    concat = {}

    # Combine two response in one batch
    for suffix, pad_val in suffix_pad.items():
        chosen = pad_to_length(batch[f"chosen_{suffix}"].to(device), max_len, pad_val, dim=1)
        rejected = pad_to_length(batch[f"rejected_{suffix}"].to(device), max_len, pad_val, dim=1)
        # (batch_num, max_len) + (batch_num, max_len) -> (2*batch_num, max_len)
        concat[f"concatenated_{suffix}"] = torch.cat([chosen, rejected], dim=0)

    return concat


@torch.no_grad()
def concatenated_forward(
    model, batch: Dict[str, torch.Tensor], pad_token_id: int, device: torch.device
):
    """Run a single forward pass over concatenated chosen+rejected sequences.

    Input:  batch with chosen (batch_num, seq_len_c) and rejected (batch_num, seq_len_r)
    Output: chosen_logps (batch_num,), rejected_logps (batch_num,),
            chosen_logits (batch_num, max_seq_len, vocab_size),
            rejected_logits (batch_num, max_seq_len, vocab_size)
    """
    concat = concatenated_inputs(batch, pad_token_id=pad_token_id, device=device)
    n = batch["chosen_input_ids"].size(0)

    # Forward: (2*batch_num, max_seq_len) -> (2*batch_num, max_seq_len, vocab_size)
    logits = model(
        input_ids=concat["concatenated_input_ids"],
        attention_mask=concat["concatenated_attention_mask"],
    ).logits

    # Per-sequence log-probs: (2*batch_num, max_seq_len, vocab_size) -> (2*batch_num,)
    logps = get_batch_logps(logits, concat["concatenated_labels"])

    # Split back into chosen and rejected
    return logps[:n], logps[n:], logits[:n], logits[n:]

In [ ]:
# --- Test get_batch_logps ---
print("-" * 20)
logits = torch.randn(2, 4, 512)           # (batch_num=2, seq_len=4, vocab_size=512)
labels = torch.tensor([[-100, 1, 2, -100], [-100, -100, 4, 5]])  # (batch_num=2, seq_len=4)
logps = get_batch_logps(logits, labels)    # Output: (batch_num=2,)
print("get_batch_logps test:", logps)
print("-" * 20)

# --- Test concatenated_inputs ---
concat_batch = concatenated_inputs(test_set, pad_token_id=0, device=torch.device("cuda"))
print("concatenated_inputs test:")
for k, v in concat_batch.items():
    print(f"  {k}: {v.size()}")
print("-" * 20)

# --- Test concatenated_forward ---
chosen_logps, rejected_logps, chosen_logits, rejected_logits = concatenated_forward(
    model, test_set, pad_token_id=0, device=torch.device("cuda")
)
print("concatenated_forward test:")
print("  chosen_logps:", chosen_logps)         # (batch_num,)
print("  rejected_logps:", rejected_logps)     # (batch_num,)
print("  chosen_logits shape:", chosen_logits.size())    # (batch_num, max_seq_len, vocab_size)
print("  rejected_logits shape:", rejected_logits.size())
print("-" * 20)

--------------------
get_batch_logps test:
tensor([-11.2851, -13.9255])
--------------------
concatenated_inputs test:
concatenated_input_ids: torch.Size([8, 25])
concatenated_attention_mask: torch.Size([8, 25])
concatenated_labels: torch.Size([8, 25])
--------------------
concatenated_forward test:
chosen_logps: tensor([ -61.7500,  -65.0000,  -81.0000, -105.5000], device='cuda:0',
       dtype=torch.bfloat16)
rejected_logps: tensor([-60.2500, -42.2500, -61.5000, -69.0000], device='cuda:0',
       dtype=torch.bfloat16)
chosen_logits shape: torch.Size([4, 25, 151936])
rejected_logits shape: torch.Size([4, 25, 151936])
--------------------


## 2.6 DPO Loss Function

In [ ]:
def dpo_loss(
    pi_chosen_logps: torch.Tensor,     # (batch_num,)
    pi_rejected_logps: torch.Tensor,   # (batch_num,)
    ref_chosen_logps: torch.Tensor,    # (batch_num,)
    ref_rejected_logps: torch.Tensor,  # (batch_num,)
    *,
    loss_type: str = "sigmoid",
    beta: float = 0.1,
    label_smoothing: float = 0.0,
    reference_free: bool = False,
):
    """Compute DPO loss and reward diagnostics.

    Input:  all logps are (batch_num,)
    Output: losses (batch_num,), chosen_rewards (batch_num,), rejected_rewards (batch_num,)
    """
    # Policy log-ratio: how much more likely chosen is vs rejected under policy
    # (batch_num,)
    pi_logratio = pi_chosen_logps - pi_rejected_logps

    # Reference log-ratio: same under reference (0 if reference_free)
    # (batch_num,) or scalar
    ref_logratio = 0.0 if reference_free else (ref_chosen_logps - ref_rejected_logps)

    # DPO logits: the gap between policy preference and reference preference
    # (batch_num,)
    logits = pi_logratio - ref_logratio

    # Compute loss based on variant
    # beta controls regularization strength (typical range: 0.1 - 0.5)
    if loss_type == "sigmoid":
        # Standard DPO: L = -log(sigmoid(beta * logits))
        # label_smoothing adds robustness to noisy preferences
        losses = (
            -F.logsigmoid(beta * logits) * (1 - label_smoothing)
            - F.logsigmoid(-beta * logits) * label_smoothing
        )
    elif loss_type == "hinge":
        losses = torch.relu(1 - beta * logits)
    elif loss_type == "ipo":
        # Identity Preference Optimization
        losses = (logits - 1.0 / (2 * beta)) ** 2
    else:
        raise ValueError(f"Unsupported loss_type: {loss_type}")

    # Reward diagnostics: KL-scaled preference signals
    # chosen_rewards should be high, rejected_rewards should be low
    # (batch_num,)
    chosen_rewards = beta * (pi_chosen_logps - ref_chosen_logps).detach()
    rejected_rewards = beta * (pi_rejected_logps - ref_rejected_logps).detach()
    return losses, chosen_rewards, rejected_rewards


def get_batch_loss_metrics(
    model, ref_model, batch, pad_token_id, device, *, beta=0.1, loss_type="sigmoid"
):
    """Compute DPO loss and diagnostic metrics for a batch (no gradient).

    Output: (scalar loss, dict of metrics)
    """
    with torch.no_grad():
        # Policy model forward: get log-probs for chosen and rejected
        pol_c_lp, pol_r_lp, pol_c_logits, pol_r_logits = concatenated_forward(
            model, batch, pad_token_id, device
        )
        # Reference model forward
        ref_c_lp, ref_r_lp, _, _ = concatenated_forward(
            ref_model, batch, pad_token_id, device
        )

    losses, chosen_rewards, rejected_rewards = dpo_loss(
        pol_c_lp, pol_r_lp, ref_c_lp, ref_r_lp, beta=beta, loss_type=loss_type
    )

    return losses.mean(), {
        "rewards/chosen": float(chosen_rewards.mean().cpu()),
        "rewards/rejected": float(rejected_rewards.mean().cpu()),
        "rewards/accuracy": float((chosen_rewards > rejected_rewards).float().mean().cpu()),
        "rewards/margin": float((chosen_rewards - rejected_rewards).mean().cpu()),
        "logps/chosen": float(pol_c_lp.mean().cpu()),
        "logps/rejected": float(pol_r_lp.mean().cpu()),
        "logits/chosen": float(pol_c_logits.mean().cpu()),
        "logits/rejected": float(pol_r_logits.mean().cpu()),
    }

In [ ]:
# --- Test loss metrics ---
loss, metrics = get_batch_loss_metrics(
    model, ref_model, test_set, pad_token_id=tokenizer.pad_token_id, device=torch.device("cuda")
)
print("Loss:", loss)
for k, v in metrics.items():
    print(f"  {k}: {v}")
print("-" * 20)

get_batch_loss_metrics test:
Loss: tensor(0.6914, device='cuda:0', dtype=torch.bfloat16)
Metrics:
  rewards/chosen: 0.0
  rewards/rejected: 0.0
  rewards/accuracy: 0.0
  rewards/margin: 0.0
  logps/chosen: -78.5
  logps/rejected: -58.25
  logits/chosen: 0.189453125
  logits/rejected: 0.33984375
--------------------


## 2.7 Training


A minimal training loop that:
* Runs policy forward WITH gradients
* Runs reference forward WITHOUT gradients
* Computes DPO loss and backpropagates

In [ ]:
def train_one_epoch(
    model, ref_model, dataloader, tokenizer, device, *, lr=5e-5, beta=0.1, loss_type="sigmoid"
) -> Dict[str, float]:
    model.train()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

    running_loss = 0.0
    last_metrics = {}
    for step, batch in enumerate(dataloader):
        batch = {k: v.to(device) for k, v in batch.items()}

        # --- Policy forward WITH gradients ---
        concat = concatenated_inputs(batch, pad_token_id=tokenizer.pad_token_id, device=device)

        # (2*batch_num, max_seq_len) -> (2*batch_num, max_seq_len, vocab_size)
        logits = model(
            input_ids=concat["concatenated_input_ids"],
            attention_mask=concat["concatenated_attention_mask"],
        ).logits

        # Extract per-sequence log-probs
        # (2*batch_num, max_seq_len, vocab_size) -> (2*batch_num,)
        logps = get_batch_logps(logits, concat["concatenated_labels"])
        n = batch["chosen_input_ids"].size(0)
        pol_c_lp, pol_r_lp = logps[:n], logps[n:]

        # --- Reference forward WITHOUT gradients ---
        with torch.no_grad():
            ref_c_lp, ref_r_lp, _, _ = concatenated_forward(
                ref_model, batch, tokenizer.pad_token_id, device
            )

        # --- DPO loss and backprop ---
        # Input: all (batch_num,) -> Output: losses (batch_num,)
        losses, _, _ = dpo_loss(
            pol_c_lp, pol_r_lp, ref_c_lp, ref_r_lp, beta=beta, loss_type=loss_type
        )
        loss = losses.mean()  # scalar

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        running_loss += float(loss.detach().cpu())

        # Collect diagnostic metrics
        _, last_metrics = get_batch_loss_metrics(
            model, ref_model, batch, tokenizer.pad_token_id, device, beta=beta, loss_type=loss_type
        )

    mean_loss = running_loss / max(1, len(dataloader))
    return {"train/mean_loss": mean_loss, **{f"last/{k}": v for k, v in last_metrics.items()}}

In [ ]:
def train_one_epoch(
    model, ref_model, dataloader, tokenizer, device, *, lr=5e-5, beta=0.1, loss_type="sigmoid"
) -> Dict[str, float]:
    model.train()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

    running_loss = 0.0
    last_metrics = {}
    for step, batch in enumerate(dataloader):
        batch = {k: v.to(device) for k, v in batch.items()}

        # --- Policy forward WITH gradients ---
        concat = concatenated_inputs(batch, pad_token_id=tokenizer.pad_token_id, device=device)

        # (2*batch_num, max_seq_len) -> (2*batch_num, max_seq_len, vocab_size)
        logits = model(
            input_ids=concat["concatenated_input_ids"],
            attention_mask=concat["concatenated_attention_mask"],
        ).logits

        # Extract per-sequence log-probs
        # (2*batch_num, max_seq_len, vocab_size) -> (2*batch_num,)
        logps = get_batch_logps(logits, concat["concatenated_labels"])
        n = batch["chosen_input_ids"].size(0)
        pol_c_lp, pol_r_lp = logps[:n], logps[n:]

        # --- Reference forward WITHOUT gradients ---
        with torch.no_grad():
            ref_c_lp, ref_r_lp, _, _ = concatenated_forward(
                ref_model, batch, tokenizer.pad_token_id, device
            )

        # --- DPO loss and backprop ---
        # Input: all (batch_num,) -> Output: losses (batch_num,)
        losses, _, _ = dpo_loss(
            pol_c_lp, pol_r_lp, ref_c_lp, ref_r_lp, beta=beta, loss_type=loss_type
        )
        loss = losses.mean()  # scalar

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        running_loss += float(loss.detach().cpu())

        # Collect diagnostic metrics
        _, last_metrics = get_batch_loss_metrics(
            model, ref_model, batch, tokenizer.pad_token_id, device, beta=beta, loss_type=loss_type
        )

    mean_loss = running_loss / max(1, len(dataloader))
    return {"train/mean_loss": mean_loss, **{f"last/{k}": v for k, v in last_metrics.items()}}


# --- Create DataLoader and Train ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def make_dataloader(dataset, tokenizer, batch_size: int = 2) -> DataLoader:
    tokenized = [
        tokenize_row(ex, tokenizer=tokenizer, max_length=256, max_prompt_length=128, max_completion_length=128)
        for ex in dataset
    ]
    return DataLoader(
        tokenized,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=DPODataCollatorWithPadding(pad_token_id=tokenizer.pad_token_id, label_pad_token_id=-100),
    )


dataset = TinyDPODataset()
dataloader = make_dataloader(dataset, tokenizer, batch_size=2)

metrics = train_one_epoch(
    model=model, ref_model=ref_model, dataloader=dataloader,
    tokenizer=tokenizer, device=device, lr=1e-4, beta=0.1, loss_type="sigmoid",
)

for k, v in metrics.items():
    if isinstance(v, float) and math.isfinite(v):
        print(f"{k}: {v:.4f}")
    else:
        print(f"{k}: {v}")

=== Demo complete ===
train/mean_loss: 0.0000
last/rewards/chosen: -0.3496
last/rewards/rejected: -23.0000
last/rewards/accuracy: 1.0000
last/rewards/margin: 22.7500
last/logps/chosen: -76.5000
last/logps/rejected: -282.0000
last/logits/chosen: -1.9219
last/logits/rejected: -1.0000


# 3) ORPO

![](https://towardsdatascience.com/wp-content/uploads/2024/06/1GoPl_kBQwWMZAGo8CcdvKA-1.png)

**(Odds Ratio Preference Optimization)**

DPO needs a frozen reference model to prevent the policy from drifting too far.
ORPO removes this requirement entirely. Instead, it combines two objectives
into a single loss:

1. **NLL loss** on the chosen completion (standard supervised fine-tuning)
2. **Odds-ratio loss** that pushes the model to prefer chosen over rejected

---

**What is the odds ratio?**

Instead of comparing raw log-probabilities like DPO does, ORPO compares
the **odds** of generating each response. If $p_c$ is the average probability
of generating the chosen completion and $p_r$ is for rejected:

$$\text{odds}(p) = \frac{p}{1 - p}$$

$$\log \text{odds\_ratio} = \log \frac{\text{odds}(p_c)}{\text{odds}(p_r)} = \underbrace{(\log p_c - \log p_r)}_{\text{direct preference}} - \underbrace{(\log(1 - p_c) - \log(1 - p_r))}_{\text{complement correction}}$$

The first part is the same as DPO's preference signal. The second part
adds a correction based on the complementary probabilities, which acts as
a built-in regularizer — no reference model needed.

---

**What changes vs DPO?**

| Component | DPO | ORPO |
|-----------|-----|------|
| Reference model | Required (frozen copy) | Not needed |
| Loss terms | $-\log\sigma(\beta \cdot \text{logits})$ | NLL(chosen) $- \beta \cdot \log\sigma(\text{log\_odds\_ratio})$ |
| Log-probs | Summed over tokens | **Averaged** over tokens |
| SFT signal | None (separate SFT stage) | Built-in via NLL term |
| Memory | 2x model weights | 1x model weights |

---

**Code changes from DPO:**

1. `orpo_loss()` replaces `dpo_loss()` — no reference log-probs needed
2. `get_batch_logps()` is called with `average_log_prob=True`
3. Training loop adds NLL loss on chosen sequences
4. No `ref_model` anywhere



## 3.1 ORPO Loss Function

In [ ]:
def orpo_loss(
    chosen_logps: torch.Tensor,
    rejected_logps: torch.Tensor,
    *,
    beta: float = 0.1,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
    """Compute ORPO odds-ratio preference loss.

    Unlike DPO, this takes NO reference model log-probs.
    The odds ratio itself acts as a self-regularizing preference signal.

    Input:
        chosen_logps:   (batch_num,) — average log-prob of chosen completions
        rejected_logps: (batch_num,) — average log-prob of rejected completions
        beta:           scalar — weight of the odds-ratio term (paper default: 0.1)

    Output:
        losses:           (batch_num,) — the OR preference loss per sample
        chosen_rewards:   (batch_num,) — monitoring signal
        rejected_rewards: (batch_num,) — monitoring signal
        mean_ratio:       scalar — mean log_sigmoid(log_odds) for logging
        mean_log_odds:    scalar — mean log_odds_ratio for logging
    """

    # --- Log odds ratio ---
    # Direct preference: how much more likely is chosen vs rejected
    # (batch_num,)
    direct = chosen_logps - rejected_logps

    # Complement correction: accounts for the "not generating" probability
    # log(1 - exp(log_p)) = log(1 - p), computed numerically stable via log1p
    # (batch_num,)
    complement = (
        torch.log1p(-torch.exp(chosen_logps))
        - torch.log1p(-torch.exp(rejected_logps))
    )

    # Full log odds ratio = direct preference - complement correction
    # (batch_num,)
    log_odds = direct - complement

    # --- Loss: beta * log_sigmoid(log_odds) ---
    # log_sigmoid squashes to [-inf, 0]
    # When log_odds is large and positive (model strongly prefers chosen) -> close to 0
    # When log_odds is negative (model prefers rejected) -> very negative
    # (batch_num,)
    ratio = F.logsigmoid(log_odds)
    losses = beta * ratio

    # --- Monitoring rewards (detached, not part of gradient) ---
    chosen_rewards = beta * chosen_logps.detach()
    rejected_rewards = beta * rejected_logps.detach()

    return losses, chosen_rewards, rejected_rewards, ratio.mean(), log_odds.mean()

In [ ]:
# --- Quick sanity check ---
# Scenario: model prefers chosen (higher log-prob)
_c = torch.tensor([-1.0, -2.0])   # chosen avg log-probs
_r = torch.tensor([-3.0, -4.0])   # rejected avg log-probs (worse)
_losses, _, _, _ratio, _log_odds = orpo_loss(_c, _r, beta=0.1)
print("ORPO loss test (model prefers chosen):")
print(f"  log_odds: {_log_odds.item():.4f} (should be positive)")
print(f"  losses:   {_losses}")
print(f"  ratio:    {_ratio.item():.4f} (closer to 0 = better)")
print()

# Scenario: model prefers rejected (bad — loss should be more negative)
_c2 = torch.tensor([-4.0, -3.0])
_r2 = torch.tensor([-1.0, -2.0])
_losses2, _, _, _ratio2, _log_odds2 = orpo_loss(_c2, _r2, beta=0.1)
print("ORPO loss test (model prefers rejected — bad):")
print(f"  log_odds: {_log_odds2.item():.4f} (should be negative)")
print(f"  losses:   {_losses2}")
print(f"  ratio:    {_ratio2.item():.4f} (more negative = worse)")

In [ ]:
def cross_entropy_loss(logits: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
    """Standard causal LM cross-entropy with next-token shift and -100 masking.

    This is the NLL term in ORPO — it provides the SFT signal on chosen sequences.
    DPO does NOT have this; it relies on a separate SFT stage before DPO training.

    Input logits: (batch_num, seq_len, vocab_size)
    Input labels: (batch_num, seq_len)
    Output: scalar loss
    """
    # Shift for next-token prediction
    # (batch_num, seq_len, vocab_size) -> (batch_num, seq_len-1, vocab_size)
    logits = logits[:, :-1, :].contiguous()
    # (batch_num, seq_len) -> (batch_num, seq_len-1)
    labels = labels[:, 1:].contiguous()

    # Flatten and compute CE, ignoring -100 (prompt tokens)
    # (batch_num * (seq_len-1), vocab_size) vs (batch_num * (seq_len-1),) -> scalar
    return nn.CrossEntropyLoss(ignore_index=-100)(
        logits.reshape(-1, logits.size(-1)),
        labels.reshape(-1),
    )

## 3.2 Training

In [ ]:
-def train_one_epoch_orpo(
    model,
    dataloader,
    tokenizer,
    device: torch.device,
    *,
    lr: float = 5e-5,
    beta: float = 0.1,
) -> Dict[str, float]:
    """One epoch of ORPO training.

    Key differences from DPO train_one_epoch:
    1. No ref_model — ORPO is reference-free
    2. Adds NLL loss on chosen sequences (built-in SFT)
    3. Uses average_log_prob=True for the odds-ratio computation
    4. Total loss = NLL(chosen) - beta * mean(log_sigmoid(log_odds))
    """
    model.train()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

    running_loss, running_nll, running_or = 0.0, 0.0, 0.0

    for step, batch in enumerate(dataloader):
        batch = {k: v.to(device) for k, v in batch.items()}
        B = batch["chosen_input_ids"].size(0)

        # --- Single forward pass over [chosen; rejected] ---
        # Same as DPO: concatenate chosen and rejected into one batch
        concat = concatenated_inputs(
            batch, pad_token_id=tokenizer.pad_token_id, device=device
        )

        # (2*B, seq_len) -> (2*B, seq_len, vocab_size)
        logits = model(
            input_ids=concat["concatenated_input_ids"],
            attention_mask=concat["concatenated_attention_mask"],
        ).logits

        # --- Term 1: NLL on chosen (the SFT signal) ---
        # DPO does NOT have this — ORPO bundles SFT into the same loss
        # Build labels: real tokens where attention=1, -100 elsewhere
        nll_labels = torch.where(
            concat["concatenated_attention_mask"] == 1,
            concat["concatenated_input_ids"],
            torch.full_like(concat["concatenated_input_ids"], -100),
        )
        # Only compute NLL on chosen (first B sequences), not rejected
        # scalar
        chosen_nll = cross_entropy_loss(logits[:B], nll_labels[:B])

        # --- Term 2: Odds-ratio preference loss ---
        # ORPO uses average_log_prob=True (unlike DPO which uses sum)
        # (2*B,) -> chosen (B,) and rejected (B,)
        all_logps = get_batch_logps(
            logits,
            concat["concatenated_labels"],
            average_log_prob=True,          # <-- key difference from DPO
        )
        chosen_logps = all_logps[:B]        # (B,)
        rejected_logps = all_logps[B:]      # (B,)

        # No ref_model log-probs needed — just pass chosen and rejected
        or_losses, c_rew, r_rew, ratio, log_odds = orpo_loss(
            chosen_logps, rejected_logps, beta=beta
        )

        # --- Combined loss ---
        # NLL pulls the model toward generating chosen text (SFT)
        # OR loss pushes the model to prefer chosen over rejected
        # The minus sign is because or_losses is already negative
        # (log_sigmoid returns values in [-inf, 0])
        loss = chosen_nll - or_losses.mean()

        # --- Backprop ---
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        running_nll += chosen_nll.item()
        running_or += or_losses.mean().item()

        print(
            f"  Step {step}: loss={loss.item():.4f}  "
            f"nll={chosen_nll.item():.4f}  "
            f"or_loss={or_losses.mean().item():.4f}  "
            f"log_odds={log_odds.item():.4f}"
        )

    n = max(1, len(dataloader))
    return {
        "train/mean_loss": running_loss / n,
        "train/mean_nll": running_nll / n,
        "train/mean_or_loss": running_or / n,
    }

In [ ]:
# Reset model (reload fresh weights so comparison is fair)
model = AutoModelForCausalLM.from_pretrained(
    model_name, torch_dtype="auto", device_map="cuda", trust_remote_code=True
)
# No ref_model needed!

dataset = TinyDPODataset()
dataloader = make_dataloader(dataset, tokenizer, batch_size=2)

print("Training one epoch of ORPO...")
print("(Notice: no ref_model parameter — ORPO is reference-free)\n")
orpo_metrics = train_one_epoch_orpo(
    model, dataloader, tokenizer, device, lr=1e-4, beta=0.1
)

print(f"\nORPO Metrics: {orpo_metrics}")

# 4) SimPO

**Simple Preference Optimization**

SimPO is the simplest variant we will see. It makes two small changes to DPO:

1. **Length normalization**: divides log-probs by sequence length, so longer
   responses are not unfairly penalized
2. **Reward margin (gamma)**: adds a target gap between chosen and rejected,
   pushing the model to be more decisive

$$\mathcal{L}_{\text{SimPO}} = -\log \sigma\!\Big(\frac{\beta}{|y_c|} \log p(y_c|x) \;-\; \frac{\beta}{|y_r|} \log p(y_r|x) \;-\; \gamma\Big)$$

where $|y_c|$ and $|y_r|$ are the lengths of chosen and rejected completions.

---

**What changes vs DPO?**

| Component | DPO | SimPO |
|-----------|-----|-------|
| Reference model | Required | **Not needed** |
| Log-probs | Summed | **Length-normalized** (averaged) |
| Margin | None | **gamma** pushes a minimum gap |
| Loss | $-\log\sigma(\beta \cdot \text{logits})$ | $-\log\sigma(\beta \cdot \text{avg\_logits} - \gamma)$ |
| Memory | 2x model weights | 1x model weights |

---

**The key insight: why length normalization matters**

Consider two chosen responses:
- Response A: 5 tokens, total log-prob = -5.0 → average = -1.0
- Response B: 50 tokens, total log-prob = -50.0 → average = -1.0

Both are equally confident per-token, but DPO (which sums) would see
Response B as much worse (-50 vs -5). SimPO fixes this by averaging.

---

**The gamma margin**

Without gamma, the model only needs chosen to be *slightly* better than rejected.
Gamma sets a minimum gap:

- gamma = 0: same as length-normalized DPO without reference model
- gamma = 0.5 (paper default): model must prefer chosen by at least 0.5 nats
- gamma = 1.0: even stronger separation required

Too high a gamma can make training unstable. The paper recommends 0.5–1.0.

---

**Code changes from DPO: literally just the loss function**

Everything else — dataset, tokenization, collator, forward pass — is identical.
We only replace `dpo_loss()` with `simpo_loss()`.



## 4.1 SimPo Loss Function

In [ ]:
def simpo_loss(
    chosen_logps: torch.Tensor,
    rejected_logps: torch.Tensor,
    *,
    beta: float = 2.0,
    gamma: float = 0.5,
    label_smoothing: float = 0.0,
    loss_type: str = "sigmoid",
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Compute SimPO loss.

    Compared to dpo_loss():
    - No ref_chosen_logps or ref_rejected_logps (reference-free)
    - Inputs should be LENGTH-NORMALIZED (average) log-probs
    - Adds gamma margin term
    - Higher default beta (2.0 vs 0.1) because averaged log-probs are smaller scale

    Input:
        chosen_logps:   (batch_num,) — average log-prob of chosen completions
        rejected_logps: (batch_num,) — average log-prob of rejected completions
        beta:           scalar — scaling factor (higher than DPO because values are averaged)
        gamma:          scalar — minimum reward margin between chosen and rejected

    Output:
        losses:           (batch_num,)
        chosen_rewards:   (batch_num,) — monitoring only
        rejected_rewards: (batch_num,) — monitoring only
    """

    # --- Core logits: length-normalized preference minus margin ---
    # In DPO this would be: pi_logratio - ref_logratio
    # In SimPO this is simply: avg_chosen - avg_rejected (no reference model)
    # Then we subtract gamma to enforce a minimum gap
    # (batch_num,)
    logits = (chosen_logps - rejected_logps) - gamma / beta

    # --- Loss computation (same variants as DPO) ---
    if loss_type == "sigmoid":
        losses = (
            -F.logsigmoid(beta * logits) * (1 - label_smoothing)
            - F.logsigmoid(-beta * logits) * label_smoothing
        )
    elif loss_type == "hinge":
        losses = torch.relu(1 - beta * logits)
    else:
        raise ValueError(f"Unsupported loss_type: {loss_type}")

    # --- Monitoring rewards (no reference model, so just scaled log-probs) ---
    chosen_rewards = beta * chosen_logps.detach()
    rejected_rewards = beta * rejected_logps.detach()

    return losses, chosen_rewards, rejected_rewards

In [ ]:
# --- Sanity check ---
# Same test scenarios as ORPO
_c = torch.tensor([-1.0, -2.0])   # chosen avg log-probs
_r = torch.tensor([-3.0, -4.0])   # rejected avg log-probs

_losses_simpo, _, _ = simpo_loss(_c, _r, beta=2.0, gamma=0.5)
print("SimPO loss test (model prefers chosen):")
print(f"  losses: {_losses_simpo}")
print(f"  (should be small — model already prefers chosen)")
print()

_losses_simpo2, _, _ = simpo_loss(_r, _c, beta=2.0, gamma=0.5)
print("SimPO loss test (model prefers rejected — bad):")
print(f"  losses: {_losses_simpo2}")
print(f"  (should be larger — model has wrong preference)")
print()

# --- Show gamma effect ---
print("Effect of gamma (margin):")
for g in [0.0, 0.5, 1.0, 2.0]:
    _l, _, _ = simpo_loss(_c, _r, beta=2.0, gamma=g)
    print(f"  gamma={g:.1f}: loss={_l.mean().item():.4f}")
print("  (higher gamma = stricter = higher loss even when model is correct)")

## 4.2 Training

In [ ]:
def train_one_epoch_simpo(
    model,
    dataloader,
    tokenizer,
    device: torch.device,
    *,
    lr: float = 5e-5,
    beta: float = 2.0,
    gamma: float = 0.5,
) -> Dict[str, float]:
    """One epoch of SimPO training.

    Key differences from DPO train_one_epoch:
    1. No ref_model — SimPO is reference-free
    2. Uses average_log_prob=True (length normalization)
    3. No NLL term (unlike ORPO) — pure preference optimization

    Key differences from ORPO train_one_epoch:
    1. No NLL/SFT term — assumes SFT was done beforehand
    2. Simpler loss — just sigmoid on normalized logits with margin
    3. Higher beta (2.0 vs 0.1) because averaged log-probs are smaller scale
    """
    model.train()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

    running_loss = 0.0

    for step, batch in enumerate(dataloader):
        batch = {k: v.to(device) for k, v in batch.items()}
        B = batch["chosen_input_ids"].size(0)

        # --- Single forward pass (same as DPO and ORPO) ---
        concat = concatenated_inputs(
            batch, pad_token_id=tokenizer.pad_token_id, device=device
        )

        logits = model(
            input_ids=concat["concatenated_input_ids"],
            attention_mask=concat["concatenated_attention_mask"],
        ).logits

        # --- Length-normalized log-probs ---
        # Same as ORPO: average_log_prob=True
        # Different from DPO: which uses sum (average_log_prob=False)
        all_logps = get_batch_logps(
            logits,
            concat["concatenated_labels"],
            average_log_prob=True,            # <-- length normalization
        )
        # (B,)
        chosen_logps = all_logps[:B]
        rejected_logps = all_logps[B:]

        # --- SimPO loss: no ref_model, no NLL, just preference with margin ---
        losses, c_rew, r_rew = simpo_loss(
            chosen_logps, rejected_logps,
            beta=beta,
            gamma=gamma,
        )
        loss = losses.mean()

        # --- Backprop ---
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        margin = (chosen_logps - rejected_logps).mean().item()
        print(
            f"  Step {step}: loss={loss.item():.4f}  "
            f"margin={margin:.4f}  "
            f"(target margin > {gamma/beta:.4f})"
        )

    return {"train/mean_loss": running_loss / max(1, len(dataloader))}

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_name, torch_dtype="auto", device_map="cuda", trust_remote_code=True
)

dataset = TinyDPODataset()
dataloader = make_dataloader(dataset, tokenizer, batch_size=2)

print("Training one epoch of SimPO...")
print("(Notice: no ref_model, no NLL term — simplest variant)\n")
simpo_metrics = train_one_epoch_simpo(
    model, dataloader, tokenizer, device, lr=1e-4, beta=2.0, gamma=0.5
)

print(f"\nSimPO Metrics: {simpo_metrics}")

# 5 Conclusion

Now that we have implemented all three methods, here is what actually differs
in code. Everything else — tokenization, collator, forward pass, dataloader —
is **identical**.

---

**Loss function comparison**

| | DPO | ORPO | SimPO |
|---|-----|------|-------|
| Reference model | Yes | No | No |
| Log-prob type | Sum | **Average** | **Average** |
| NLL/SFT term | No | **Yes** | No |
| Margin (gamma) | No | No | **Yes** |
| Typical beta | 0.1 | 0.1 | **2.0** |
| Memory (models) | 2x | 1x | 1x |

---

**When to use which?**

| Scenario | Method | Reason |
|----------|--------|--------|
| Have a good base model + GPU memory for 2 models | DPO | Strongest theoretical guarantees |
| Want to SFT and align in a single pass | ORPO | Built-in NLL saves a training stage |
| Memory constrained, model already SFT'd | SimPO | Simplest, no ref model, no NLL |
| Responses vary a lot in length | SimPO | Length normalization handles this |

---